# add-sub-div-back-lambdas — ex2: dispatch through BACK for a 2-op mini reverse pass

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `add-sub-div-back-lambdas`. Running the final beacon cell reports progress against the `Backprop: add/sub/div back as lambdas` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: add/sub/div back as lambdas` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`add-sub-div-back-lambdas`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "add-sub-div-back-lambdas"
DD_SUBTOPIC = "Backprop: add/sub/div back as lambdas"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `add`/`sub`/`div` back lambdas in a mini dispatcher — quick refresher

ex1 built the 6-entry `BACK` dict and called lambdas directly. The deeper facet is **dispatch-as-data**: the reverse pass never names a back fn — it always looks one up by `(op_name, argnum)` and calls it.

```python
for argnum, parent in node.recipe.parents.items():
    back_fn = BACK[(node.recipe.func_name, argnum)]
    grad_in = back_fn(grad_out, node.array, *node.recipe.args)
    grads[parent] = grads.get(parent, 0) + grad_in
```

Two invariants:
- **The dispatcher routes through the dict EVERY call.** If you hardcode `g + 0` inside the loop for add, swapping `BACK[('add', 0)]` to a scaled lambda silently does nothing. Real reverse passes go through the table.
- **Symmetric ops still get two entries.** `('add', 0)` and `('add', 1)` have identical bodies. The dispatcher cannot know — it just keys by `(op, argnum)`.

### Exercise 2 — dispatch through BACK for a 2-op mini reverse pass

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply BACK-dict dispatch in a 2-op mini reverse pass: route grad_out through (op_name, argnum) lookups for `z = (x - y) / w`.
> Keywords: dispatch, back-dict, reverse-pass, argnum
> ```

**KCs targeted:** `arg-position-back-functions`, `backward-fn-signature`

Build a single dispatcher `mini_back(op_name, argnum, grad_out, out, x, y)` that looks up `BACK[(op_name, argnum)]` and calls it. Then use that dispatcher to walk the reverse pass of `z = (x - y) / w` and produce `dL/dx`, `dL/dy`, `dL/dw`.

BACK contains the 6 lambdas from ex1 (`add`, `sub`, `div` x argnums 0, 1). The dispatcher does NOT name any back fn — it always indexes into BACK.

Algorithm for the 2-op chain `s = x - y; z = s / w` (no broadcasting):

```python
# 1. seed grad_out_z = ones_like(z)
# 2. dispatch div argnum=0 → grad_s = mini_back('div', 0, grad_out_z, z, s, w)
# 3. dispatch div argnum=1 → grad_w = mini_back('div', 1, grad_out_z, z, s, w)
# 4. dispatch sub argnum=0 → grad_x = mini_back('sub', 0, grad_s, s, x, y)
# 5. dispatch sub argnum=1 → grad_y = mini_back('sub', 1, grad_s, s, x, y)
```

Closed-form check (you'll see these in the tests):
```
dL/dx =  1/w           # +1 from sub * 1/w from div
dL/dy = -1/w           # -1 from sub * 1/w from div
dL/dw = -s / w**2      # -s/w**2 directly from div argnum=1
```

Implement `mini_back` AND `reverse_div_sub_chain(x, y, w)`. Return a dict `{'dx': ..., 'dy': ..., 'dw': ...}`. No autograd.

In [ ]:
# BACK is provided — same 6 lambdas as ex1.
BACK = {
    ('add', 0): lambda g, o, x, y:  g,
    ('add', 1): lambda g, o, x, y:  g,
    ('sub', 0): lambda g, o, x, y:  g,
    ('sub', 1): lambda g, o, x, y: -g,
    ('div', 0): lambda g, o, x, y:  g / y,
    ('div', 1): lambda g, o, x, y: -g * x / (y * y),
}


def mini_back(op_name: str, argnum: int, grad_out, out, x, y):
    """Look up and call BACK[(op_name, argnum)]. No hardcoding."""
    raise NotImplementedError()


def reverse_div_sub_chain(x, y, w):
    """Walk the reverse pass of z = (x - y) / w. Return dict dx/dy/dw."""
    raise NotImplementedError()


def _test_ex2():
    # --- dispatcher routes through BACK ---
    x = t.tensor([1.0, 2.0])
    y = t.tensor([3.0, 4.0])
    g = t.ones(2)
    out_add = x + y
    assert t.allclose(mini_back('add', 0, g, out_add, x, y), g)
    assert t.allclose(mini_back('add', 1, g, out_add, x, y), g)
    out_sub = x - y
    assert t.allclose(mini_back('sub', 0, g, out_sub, x, y),  g)
    assert t.allclose(mini_back('sub', 1, g, out_sub, x, y), -g)

    # --- dispatcher re-reads BACK each call (proves no hardcoding) ---
    saved = BACK[('add', 0)]
    BACK[('add', 0)] = lambda g, o, x, y: g * 7.0
    try:
        res = mini_back('add', 0, g, x + y, x, y)
        assert t.allclose(res, g * 7.0), f'dispatcher must re-read BACK: {res}'
    finally:
        BACK[('add', 0)] = saved

    # --- 2-op chain: z = (x - y) / w ---
    x = t.tensor([6.0, 10.0, 14.0])
    y = t.tensor([2.0,  4.0,  6.0])
    w = t.tensor([2.0,  2.0,  4.0])
    grads = reverse_div_sub_chain(x, y, w)
    assert set(grads.keys()) == {'dx', 'dy', 'dw'}, f'keys: {grads.keys()}'

    # Closed-form: with grad_out_z = ones:
    expected_dx =  1.0 / w
    expected_dy = -1.0 / w
    s = x - y
    expected_dw = -s / (w * w)
    assert t.allclose(grads['dx'], expected_dx), f'dx: {grads["dx"]} vs {expected_dx}'
    assert t.allclose(grads['dy'], expected_dy), f'dy: {grads["dy"]} vs {expected_dy}'
    assert t.allclose(grads['dw'], expected_dw), f'dw: {grads["dw"]} vs {expected_dw}'

    # --- agreement with torch.autograd ---
    x_r = x.clone().requires_grad_(True)
    y_r = y.clone().requires_grad_(True)
    w_r = w.clone().requires_grad_(True)
    ((x_r - y_r) / w_r).sum().backward()
    assert t.allclose(grads['dx'], x_r.grad, atol=1e-6)
    assert t.allclose(grads['dy'], y_r.grad, atol=1e-6)
    assert t.allclose(grads['dw'], w_r.grad, atol=1e-6)

    # --- swap BACK[('div', 1)] to scaled — chain output must REFLECT the swap ---
    saved_div1 = BACK[('div', 1)]
    BACK[('div', 1)] = lambda g, o, a, b: 2.0 * (-g * a / (b * b))
    try:
        grads2 = reverse_div_sub_chain(x, y, w)
        assert t.allclose(grads2['dw'], 2.0 * expected_dw), (
            f'chain did not dispatch through BACK on call to div argnum=1: '
            f'got {grads2["dw"]}, expected {2.0 * expected_dw}'
        )
    finally:
        BACK[('div', 1)] = saved_div1
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def mini_back(op_name: str, argnum: int, grad_out, out, x, y):
    return BACK[(op_name, argnum)](grad_out, out, x, y)


def reverse_div_sub_chain(x, y, w):
    s = x - y
    z = s / w
    grad_z = t.ones_like(z)
    grad_s = mini_back('div', 0, grad_z, z, s, w)
    grad_w = mini_back('div', 1, grad_z, z, s, w)
    grad_x = mini_back('sub', 0, grad_s, s, x, y)
    grad_y = mini_back('sub', 1, grad_s, s, x, y)
    return {'dx': grad_x, 'dy': grad_y, 'dw': grad_w}
```

**Why route through BACK every call.** In the test above, re-binding `BACK[('div', 1)]` to a doubled lambda must change `grads2['dw']`. If `reverse_div_sub_chain` had hardcoded `-grad_z * s / (w * w)` inline, the swap would do nothing — proof that dispatch-as-data is the right abstraction.

**Why two separate dispatches for div.** `(div, 0)` and `(div, 1)` are NOT the same function — symmetric ops like `add` happen to share a body, but `div` doesn't. The dispatcher cannot collapse them; it must look up each argnum independently.

**Chain shape.** `grad_s` is the intermediate gradient feeding back into the sub op. It is itself the OUTPUT of div's argnum=0 back fn — exactly how `Recipe.parents` would route it in a real reverse pass.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()